# Ingest

load pdf, chunk documents, convert into embeddings, store in ChromaDB

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import plotly.graph_objects as go

In [11]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
OPENAI_MODEL = "gpt-4.1-nano"
VECTOR_DB_PATH = "../irc_vectordb"

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print("OpenAI API key found!")
else:
    print("OpenAI API key not set")



OpenAI API key found!


In [ ]:
# Load pdf - took ~25 sec for me the first time I ran this cell
# Im on a Macbook Pro M1 8GB 2020 for reference

IRC_path = '../Internal Revenue Code/Internal Revenue Code.pdf'
loader = PyMuPDFLoader(IRC_path)
documents = loader.load()

In [9]:
documents[100]

Document(metadata={'producer': 'iText 2.0.8 (by lowagie.com)', 'creator': '', 'creationdate': '2026-01-12T07:28:23-05:00', 'source': '../Internal Revenue Code/Internal Revenue Code.pdf', 'file_path': '../Internal Revenue Code/Internal Revenue Code.pdf', 'total_pages': 7164, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2026-01-12T07:28:23-05:00', 'trapped': '', 'modDate': "D:20260112072823-05'00'", 'creationDate': "D:20260112072823-05'00'", 'page': 100}, page_content='Except as otherwise provided in this paragraph, the term "qualified foreign corporation"\nmeans any foreign corporation if—\n(I) such corporation is incorporated in a possession of the United States, or\n(II) such corporation is eligible for benefits of a comprehensive income tax treaty with\nthe United States which the Secretary determines is satisfactory for purposes of this\nparagraph and which includes an exchange of information program.\n(ii) Dividends on stock readily tra

In [13]:
# Chunk documents - chunk_size = 1000 & chunk_overlap = 200 yielded ~34k chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Number of Chunks: {len(chunks)} chunks")

Number of Chunks: 33972 chunks


In [ ]:
# Create embeddings and store - took ~6 min

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

if os.path.exists(VECTOR_DB_PATH):
    Chroma(persist_directory=VECTOR_DB_PATH, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=VECTOR_DB_PATH)

/Users/devpatel/projects/IRC_Chatbot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8807.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
